# ResNet-50 CNN — CUB-200-2011 Bird Photos (R4)

This notebook fine-tunes an **ImageNet-pretrained ResNet-50** to classify the 200 bird species from their photos, and compares it with the attribute-based models (decision tree, random forest, KNN, logistic regression, MLP).

**Main metric:** macro-F1 on the official test split (accuracy and top-5 accuracy are reported alongside).

**Input:** the cropped photos made by the team's image pipeline (`imagePrep.ipynb` → `scripts/prepare_images.py`). The notebook runs both steps itself if their output is missing. Following *Decision 9* in [`05_image_preparation.ipynb`](05_image_preparation.ipynb), every photo (train **and** test) is cropped to its bounding box plus a 10% margin. The report should say so, because the boxes are extra information that a plain photo would not have.

**Protocol (no test leakage):**
- validation set = 20% of the official **training** photos, stratified by species (`make_splits` in `scripts/augmentation.py`);
- each epoch is scored on validation, and the epoch with the best validation macro-F1 is kept;
- the official test photos are used **once**, at the end (section 7).

### This is the test run (Colab T4, 224 px)
A short, cheap run to check the code and get a first result before the full run (Colab L4, 448 px). Set `PRETRAINED = False` in section 1 to train the same network **from scratch** for comparison.

| Setting | Pretrained (default) | From scratch |
|---|---|---|
| Weights | torchvision `IMAGENET1K_V1` | random |
| Epochs | 15 | 60 |
| Learning rate (SGD, momentum 0.9) | backbone 0.002, new classifier 0.02 | 0.025 everywhere |
| Schedule | 1 warm-up epoch, then cosine decay | 3 warm-up epochs, then cosine decay |
| Expected time on a T4 | ~10 min | ~40 min |

### How to run on Colab
1. Open this notebook from GitHub in Colab (File → Open notebook → GitHub → `Flamingsides/f20dl-coursework-1`, branch `Ryan`).
2. Runtime → Change runtime type → **T4 GPU**.
3. Put your `kaggle.json` (kaggle.com → Settings → API → Create New Token) in Google Drive at `MyDrive/f20dl_resnet50/kaggle.json`. It is only needed the first time.
4. Runtime → Run all. The first session downloads and crops the photos (~10 min) and caches them in Drive. Later sessions restore that cache in under a minute.

If Colab disconnects, run all again: training resumes from the last epoch saved in Drive.

## 1. Settings

In [ ]:
# --- GenAI disclosure (F20DL coursework spec, Section 8) ---
# This notebook was drafted with Claude (Anthropic) in a guided conversation, then reviewed,
# run and checked by Ryan. Main prompts used:
# 1. "I want to create a CNN model that uses the ResNet-50 network. Help me research how it can be
#     best implemented and create a plan for it."
# 2. "I need it to achieve the best accuracy possible based on the F1 score of the model"
# 3. "It can be a fine-tuned version of a pretrained ResNet-50 but give some options and also compare it
#     to an estimation of how it would compare to the accuracy of one trained from scratch."
# 4. "let's stick to your plan of using the T4 for testing and the L4 for the full run.
#     You may start with the code for the test run now"

PRETRAINED = True      # False = same network trained from scratch (comparison run)
SMOKE_TEST = False     # True = tiny subset, 1 epoch: only checks that the code runs end to end

REPO_URL = "https://github.com/Flamingsides/f20dl-coursework-1.git"
BRANCH = "Ryan"
DRIVE_FOLDER = "f20dl_resnet50"   # folder in MyDrive for the data cache, checkpoints and results

IMG_SIZE = 224
BATCH_SIZE = 64
LABEL_SMOOTHING = 0.1
WEIGHTS = "IMAGENET1K_V1"   # the standard ImageNet ResNet-50 weights used in most CUB papers

if PRETRAINED:
    EPOCHS, WARMUP_EPOCHS = 15, 1
    LR_BACKBONE = 1e-3 * BATCH_SIZE / 32   # small steps: the ImageNet features only need adjusting
    LR_HEAD = 10 * LR_BACKBONE              # the new 200-way classifier starts from random, so it learns faster
    WEIGHT_DECAY = 1e-4
else:
    EPOCHS, WARMUP_EPOCHS = 60, 3
    LR_BACKBONE = LR_HEAD = 0.1 * BATCH_SIZE / 256   # standard ImageNet-from-scratch rate, scaled to the batch
    WEIGHT_DECAY = 5e-4                              # more regularisation: ~24 photos per species is very little

if SMOKE_TEST:
    EPOCHS, WARMUP_EPOCHS, BATCH_SIZE = 1, 0, 8

RUN_NAME = f"resnet50_{'pretrained' if PRETRAINED else 'scratch'}_{IMG_SIZE}px" + ("_smoke" if SMOKE_TEST else "")
CONFIG = dict(run=RUN_NAME, pretrained=PRETRAINED, weights=WEIGHTS if PRETRAINED else None, img_size=IMG_SIZE,
              batch_size=BATCH_SIZE, epochs=EPOCHS, warmup_epochs=WARMUP_EPOCHS, lr_backbone=LR_BACKBONE,
              lr_head=LR_HEAD, weight_decay=WEIGHT_DECAY, label_smoothing=LABEL_SMOOTHING, smoke_test=SMOKE_TEST)
CONFIG

## 2. Environment

On Colab: mount Google Drive and clone the repository (for the team's image-pipeline code). Locally: use this repository.

In [ ]:
import json
import math
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive") / DRIVE_FOLDER
    DRIVE.mkdir(parents=True, exist_ok=True)
    ROOT = Path("/content/f20dl-coursework-1")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(ROOT)], check=True)
    OUT = DRIVE                                    # survives disconnects
    CKPT_DIR = DRIVE / "checkpoints"
else:
    DRIVE = None
    ROOT = Path.cwd() if (Path.cwd() / "notebooks").exists() else Path.cwd().parent
    OUT = ROOT / "results" / "resnet50"
    CKPT_DIR = ROOT / "data" / "processed" / "checkpoints"   # git-ignored: checkpoints are ~100-200 MB

FIGURES = OUT / "figures"
for d in (OUT, CKPT_DIR, FIGURES):
    d.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "scripts"))

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    torch.backends.cudnn.benchmark = True          # picks the fastest convolution algorithms for fixed-size inputs
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")
USE_AMP = DEVICE.type == "cuda"                    # mixed precision: ~2x faster on a T4, same accuracy

print(f"Colab: {IN_COLAB} | device: {DEVICE}" + (f" ({torch.cuda.get_device_name()})" if DEVICE.type == "cuda" else ""))
print(f"torch {torch.__version__} | outputs -> {OUT}")
if IN_COLAB and DEVICE.type != "cuda":
    raise RuntimeError("No GPU: Runtime -> Change runtime type -> T4 GPU, then run all again.")

## 3. Data

The cropped photos are made by the team's image pipeline: `imagePrep.ipynb` checks every photo and writes `data/cub_images.csv`, then `scripts/prepare_images.py` converts to RGB, crops to the bounding box + 10% margin and resizes the shorter side to 256 px. On Colab, the result is cached in Drive as a zip, so this only runs once.

In [ ]:
PROCESSED = ROOT / "data" / "processed"
CSV = PROCESSED / "cub_images_cropped.csv"
CACHE_ZIP = DRIVE / "cub_processed_256px.zip" if IN_COLAB else None

def run(cmd):
    print("$", " ".join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), check=True, cwd=ROOT)

if not CSV.exists() and IN_COLAB and CACHE_ZIP.exists():
    print("Restoring cropped photos from the Drive cache...")
    shutil.unpack_archive(CACHE_ZIP, ROOT / "data")

if not CSV.exists():
    has_raw = any((ROOT / "data" / "raw").rglob("image_class_labels.txt")) or (ROOT / "CUB_200_2011").exists()
    if not has_raw:
        if IN_COLAB:
            token = DRIVE / "kaggle.json"
            if not token.exists():
                raise FileNotFoundError(f"Put your kaggle.json in Google Drive at {token} (see the top of this notebook)")
            kaggle_dir = Path.home() / ".kaggle"
            kaggle_dir.mkdir(exist_ok=True)
            shutil.copy(token, kaggle_dir / "kaggle.json")
            os.chmod(kaggle_dir / "kaggle.json", 0o600)
        run([sys.executable, "scripts/download_data.py"])
    run(["jupyter", "nbconvert", "--to", "notebook", "--execute", "--output-dir", PROCESSED / "_executed",
         "notebooks/imagePrep.ipynb"])
    run([sys.executable, "scripts/prepare_images.py"])
    if IN_COLAB:
        print("Caching the cropped photos in Drive...")
        shutil.make_archive(str(CACHE_ZIP.with_suffix("")), "zip", root_dir=ROOT / "data", base_dir="processed")

print(f"{CSV.relative_to(ROOT)}: {len(pd.read_csv(CSV)):,} photos")

## 4. Splits, augmentation and loaders

The splits and dataset class come from `scripts/augmentation.py`, so this model uses exactly the same train / validation / test photos as the rest of the image pipeline. The transforms repeat that file's choices (random crop, horizontal flip, ±15° rotation, lighting jitter with almost no hue change, random erasing) but take the image size as a parameter, so the same code serves the 448 px full run.

In [ ]:
from torchvision.transforms import v2 as T
from augmentation import BirdImages, make_splits, IMAGENET_MEAN, IMAGENET_STD, N_CLASSES, RANDOM_STATE

def make_transforms(size):
    """Same augmentation choices as scripts/augmentation.py, for any input size."""
    train = T.Compose([
        T.ToImage(),
        T.RandomResizedCrop(size, scale=(0.6, 1.0), ratio=(3 / 4, 4 / 3), antialias=True),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=15),
        T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.02),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        T.RandomErasing(p=0.25, scale=(0.02, 0.15)),
    ])
    evaluate = T.Compose([                                     # validation and test: no randomness
        T.ToImage(),
        T.Resize(round(size * 256 / 224), antialias=True),     # 224 -> 256, as in augmentation.py
        T.CenterCrop(size),
        T.ToDtype(torch.float32, scale=True),
        T.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])
    return train, evaluate

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

train_df, val_df, test_df = make_splits(CSV)
if SMOKE_TEST:   # 10 species, a few photos each
    pick = lambda df, n: df[df.class_id <= 10].groupby("class_id").head(n)
    train_df, val_df, test_df = pick(train_df, 4), pick(val_df, 2), pick(test_df, 3)
assert not set(train_df.image_id) & set(val_df.image_id) and not set(train_df.image_id) & set(test_df.image_id)

train_tf, eval_tf = make_transforms(IMG_SIZE)
NUM_WORKERS = 0 if SMOKE_TEST else min(8, os.cpu_count() or 2)
loader_args = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
                   persistent_workers=NUM_WORKERS > 0)
train_loader = DataLoader(BirdImages(train_df, PROCESSED, train_tf), shuffle=True, drop_last=True,
                          generator=torch.Generator().manual_seed(RANDOM_STATE), **loader_args)
val_loader = DataLoader(BirdImages(val_df, PROCESSED, eval_tf), **loader_args)
test_loader = DataLoader(BirdImages(test_df, PROCESSED, eval_tf), **loader_args)

species = (pd.read_csv(CSV)[["class_id", "species"]].drop_duplicates()
           .set_index("class_id").sort_index().species.tolist())       # index = class_id - 1
print(f"train {len(train_df):,} | val {len(val_df):,} | test {len(test_df):,} photos | "
      f"{len(train_loader)} batches of {BATCH_SIZE} per epoch | {NUM_WORKERS} loader workers")

## 5. Model, optimiser and schedule

- **ResNet-50** with its 1000-class ImageNet classifier replaced by a new 200-class layer.
- **Two learning rates:** the pretrained layers move 10× more slowly than the new classifier, so fine-tuning adjusts the ImageNet features instead of overwriting them.
- **SGD with momentum 0.9**, a short linear warm-up, then cosine decay to zero (updated every batch).
- **Label smoothing 0.1:** the target is 90% on the true species and the rest spread over the others. With ~24 photos per species this discourages over-confident memorising and usually adds about a point on CUB.

In [ ]:
from torchvision.models import resnet50, ResNet50_Weights

model = resnet50(weights=ResNet50_Weights[WEIGHTS] if PRETRAINED else None, progress=False)
model.fc = nn.Linear(model.fc.in_features, N_CLASSES)
model = model.to(DEVICE, memory_format=torch.channels_last)    # faster convolutions on tensor-core GPUs

head_params = list(model.fc.parameters())
head_ids = {id(p) for p in head_params}
backbone_params = [p for p in model.parameters() if id(p) not in head_ids]
optimizer = torch.optim.SGD([{"params": backbone_params, "lr": LR_BACKBONE},
                             {"params": head_params, "lr": LR_HEAD}],
                            momentum=0.9, nesterov=True, weight_decay=WEIGHT_DECAY)

STEPS = EPOCHS * len(train_loader)
WARMUP_STEPS = WARMUP_EPOCHS * len(train_loader)
def lr_factor(step):
    if step < WARMUP_STEPS:
        return (step + 1) / WARMUP_STEPS
    progress = (step - WARMUP_STEPS) / max(1, STEPS - WARMUP_STEPS)
    return 0.5 * (1 + math.cos(math.pi * progress))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)

print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters | {STEPS} training steps")

## 6. Training (resumable)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, top_k_accuracy_score

def autocast():
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP)

def train_one_epoch():
    model.train()
    loss_sum, correct, seen = 0.0, 0, 0
    for x, y in train_loader:
        x = x.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        y = y.to(DEVICE, non_blocking=True)
        with autocast():
            logits = model(x)
            loss = criterion(logits, y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        loss_sum += loss.item() * len(y)
        correct += (logits.argmax(1) == y).sum().item()
        seen += len(y)
    return loss_sum / seen, correct / seen

@torch.no_grad()
def predict(loader):
    """Class probabilities, true labels (0-199) and mean cross-entropy (without smoothing)."""
    model.eval()
    probs, labels, loss_sum = [], [], 0.0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True, memory_format=torch.channels_last)
        y = y.to(DEVICE, non_blocking=True)
        with autocast():
            logits = model(x)
        loss_sum += nn.functional.cross_entropy(logits.float(), y, reduction="sum").item()
        probs.append(logits.float().softmax(1).cpu())
        labels.append(y.cpu())
    probs, labels = torch.cat(probs).numpy(), torch.cat(labels).numpy()
    return probs, labels, loss_sum / len(labels)

def scores(probs, labels):
    preds = probs.argmax(1)
    return {"accuracy": accuracy_score(labels, preds),
            "macro_f1": f1_score(labels, preds, average="macro"),
            "top5_accuracy": top_k_accuracy_score(labels, probs, k=5, labels=np.arange(N_CLASSES))}

def save_atomic(obj, path):
    tmp = path.with_suffix(".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)

LAST, BEST = CKPT_DIR / f"{RUN_NAME}_last.pt", CKPT_DIR / f"{RUN_NAME}_best.pt"
start_epoch, history, best_f1 = 0, [], -1.0
if LAST.exists():
    ckpt = torch.load(LAST, map_location=DEVICE, weights_only=False)
    if ckpt["config"] != CONFIG:
        raise ValueError(f"{LAST.name} was saved with different settings - delete it or change RUN_NAME")
    model.load_state_dict(ckpt["model"])
    optimizer.load_state_dict(ckpt["optimizer"])
    scheduler.load_state_dict(ckpt["scheduler"])
    scaler.load_state_dict(ckpt["scaler"])
    start_epoch, history, best_f1 = ckpt["epoch"] + 1, ckpt["history"], ckpt["best_f1"]
    print(f"Resuming {RUN_NAME} after epoch {start_epoch} (best val macro-F1 so far {best_f1:.4f})")

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()
    train_loss, train_acc = train_one_epoch()
    val_probs, val_labels, val_loss = predict(val_loader)
    val = scores(val_probs, val_labels)
    history.append({"epoch": epoch + 1, "train_loss": train_loss, "train_accuracy": train_acc,
                    "val_loss": val_loss, **{f"val_{k}": v for k, v in val.items()},
                    "seconds": time.time() - t0})
    improved = val["macro_f1"] > best_f1
    if improved:
        best_f1 = val["macro_f1"]
        save_atomic({"model": model.state_dict(), "epoch": epoch + 1, "val": val, "config": CONFIG}, BEST)
    save_atomic({"model": model.state_dict(), "optimizer": optimizer.state_dict(),
                 "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), "epoch": epoch,
                 "history": history, "best_f1": best_f1, "config": CONFIG}, LAST)
    print(f"epoch {epoch + 1:>2}/{EPOCHS} | train loss {train_loss:.3f} acc {train_acc:.3f} | "
          f"val loss {val_loss:.3f} acc {val['accuracy']:.3f} macro-F1 {val['macro_f1']:.3f} "
          f"top-5 {val['top5_accuracy']:.3f} | {time.time() - t0:.0f}s" + ("  *best" if improved else ""))

history_df = pd.DataFrame(history)
history_df.to_csv(OUT / f"{RUN_NAME}_history.csv", index=False)
print(f"Total training time: {history_df.seconds.sum() / 60:.1f} min")

### Learning curves

A widening gap between training and validation accuracy means the network is memorising the training photos (overfitting). Validation macro-F1 levelling off shows where more epochs stop helping.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
axes[0].plot(history_df.epoch, history_df.train_loss, label="train (with label smoothing)")
axes[0].plot(history_df.epoch, history_df.val_loss, label="validation")
axes[0].set(title="Loss", xlabel="epoch", ylabel="cross-entropy")
axes[1].plot(history_df.epoch, history_df.train_accuracy, label="train accuracy (augmented photos)")
axes[1].plot(history_df.epoch, history_df.val_accuracy, label="validation accuracy")
axes[1].plot(history_df.epoch, history_df.val_macro_f1, "--", label="validation macro-F1")
axes[1].set(title="Accuracy and macro-F1", xlabel="epoch", ylim=(0, 1))
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.suptitle(RUN_NAME, fontweight="bold")
plt.tight_layout()
fig.savefig(FIGURES / f"{RUN_NAME}_curves.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Final test evaluation (run once)

The weights from the epoch with the best **validation** macro-F1 are scored once on the 5,794 official test photos.

In [ ]:
best = torch.load(BEST, map_location=DEVICE, weights_only=False)
model.load_state_dict(best["model"])
test_probs, test_labels, test_loss = predict(test_loader)
test = scores(test_probs, test_labels)

results = {"run": RUN_NAME, "config": CONFIG, "best_epoch": best["epoch"], "val": best["val"], "test": test,
           "test_photos": int(len(test_labels)), "train_minutes": round(history_df.seconds.sum() / 60, 1)}
(OUT / f"{RUN_NAME}_results.json").write_text(json.dumps(results, indent=2))

per_class = pd.DataFrame({"class_id": np.arange(1, N_CLASSES + 1), "species": species,
                          "f1": f1_score(test_labels, test_probs.argmax(1), average=None, labels=np.arange(N_CLASSES))})
per_class.to_csv(OUT / f"{RUN_NAME}_per_class_f1.csv", index=False)

# Best attribute-based results on the same official test split (from the notebooks named)
comparison = pd.DataFrame([
    ("Random forest, tuned (randomForest.ipynb)", "312 attributes", 0.4563, 0.4349),
    ("RF + logistic regression ensemble (randomForestOptimisation.ipynb)", "attributes + aggregates", 0.5104, 0.5059),
    (f"ResNet-50, {'pretrained' if PRETRAINED else 'from scratch'}, {IMG_SIZE} px (this notebook)", "photos",
     test["accuracy"], test["macro_f1"]),
], columns=["model", "input", "test accuracy", "test macro-F1"])
print(f"Best epoch {best['epoch']} | test accuracy {test['accuracy']:.4f} | macro-F1 {test['macro_f1']:.4f} | "
      f"top-5 {test['top5_accuracy']:.4f}")
comparison.round(4)

## 8. Which species does it confuse?

The most frequent test mistakes, as (true species → predicted species). On CUB these are usually sibling species from the same family, which matches what the attribute models found.

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(test_labels, test_probs.argmax(1), labels=np.arange(N_CLASSES))
np.fill_diagonal(cm, 0)
top = np.dstack(np.unravel_index(np.argsort(cm, axis=None)[::-1][:10], cm.shape))[0]
pd.DataFrame([(species[t], species[p], cm[t, p], int((test_labels == t).sum())) for t, p in top],
             columns=["true species", "predicted as", "mistakes", "test photos of true species"])

## 9. Next steps

1. **From-scratch comparison:** set `PRETRAINED = False` in section 1 and run all again (~40 min on a T4). Its results are saved under a different run name, so both can be compared.
2. **Full run (Colab L4, 448 px):** needs the photos stored larger (shorter side ~512 px instead of 256), then `IMG_SIZE = 448` and around 30 epochs. That run is not set up yet.